# ibench on Colab: the x86 runs

Runs the **cloud subset** of the benchmark (Qwen2.5-1.5B-Instruct Q4_K_M, all physical cores, 200 and 4,000-token
prompts, 1/4/8 concurrent requests, 3 repeats) on a Colab CPU, for comparison with the Apple Silicon runs.

1. **Runtime → Change runtime type → CPU.** No GPU: this measures the CPU. A free CPU runtime has 2 vCPUs (one
   physical core); the High-RAM CPU runtime has more cores and gives a fairer comparison with a 4-performance-core laptop.
2. **Runtime → Run all.** Building llama.cpp takes about 5–10 minutes; the sweep takes from about 30 minutes (several
   cores) to a few hours (one core). Keep the tab open so Colab doesn't disconnect.
3. The last cell downloads a zip of the results. Import it on the dashboard's **Compare** page, or unzip it into
   `results/` in the repo.

llama.cpp is built from tag `b12359`, the same build used for the laptop runs, so only the hardware differs.

In [ ]:
# What machine did Colab give us?
!lscpu | grep -E "Model name|^CPU\(s\)|Thread\(s\) per core|Core\(s\) per socket|Socket|Flags" | cut -c1-200
!free -g | head -2
import os, psutil
physical = psutil.cpu_count(logical=False)
print(f"\nphysical cores: {physical}, logical CPUs: {os.cpu_count()}")
if physical < 4:
    print("Note: fewer than 4 physical cores. The results are valid, but a High-RAM CPU runtime gives a fairer "
          "comparison with a laptop's 4 performance cores.")

## Settings

In [ ]:
BRANCH = "main"            # branch of the ibench repo to use
LLAMA_TAG = "b12359"       # llama.cpp release tag; b12359 matches the laptop runs
PRESET = "cloud"           # see ibench/presets.py
HOURLY_COST = None         # USD per hour for cost per 1M tokens; Colab has no per-hour price, so leave None unless you map it to one
LABEL = ""                 # optional; defaults to "Colab x86: <CPU model>"

## 1. Get ibench

In [ ]:
%cd /content
!rm -rf inference_tools && git clone -q --depth 1 -b {BRANCH} https://github.com/a-ramakrishnan/inference_tools
%cd /content/inference_tools
!pip install -q -e .

## 2. Build llama-server (CPU, native instructions)

In [ ]:
%cd /content
!test -d llama.cpp || git clone -q --depth 1 --branch {LLAMA_TAG} https://github.com/ggml-org/llama.cpp
!cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DGGML_NATIVE=ON -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF > /tmp/cmake.log 2>&1 || tail -30 /tmp/cmake.log
!cmake --build llama.cpp/build -j$(nproc) --target llama-server > /tmp/build.log 2>&1 || tail -30 /tmp/build.log
LLAMA_SERVER = "/content/llama.cpp/build/bin/llama-server"
os.environ["LLAMA_SERVER"] = LLAMA_SERVER     # ibench reads this
!{LLAMA_SERVER} --version
%cd /content/inference_tools

## 3. Download the model (about 1.1 GB)

In [ ]:
!mkdir -p models
!test -f models/qwen2.5-1.5b-instruct-q4_k_m.gguf || curl -sL -o models/qwen2.5-1.5b-instruct-q4_k_m.gguf https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf
!ls -lh models

## 4. Smoke test (under a minute)

In [ ]:
!ibench info | head -30
!ibench run --preset smoke --models qwen2.5-1.5b-instruct-q4_k_m.gguf --label "Colab smoke test"

## 5. The sweep

In [ ]:
import json, subprocess
from ibench.hardware import detect
hw = detect()
label = LABEL or f"Colab x86: {hw['cpu']}"
args = ["ibench", "run", "--preset", PRESET, "--label", label]
if HOURLY_COST is not None:
    args += ["--hourly-cost", str(HOURLY_COST)]
print(" ".join(args))
subprocess.run(args, check=True)

## 6. Download the results

In [ ]:
import glob, shutil
from google.colab import files
sweeps = sorted(p for p in glob.glob("results/*/meta.json") if json.load(open(p)).get("spec", {}).get("label") == label)
sid = sweeps[-1].split("/")[1]
print("sweep:", sid)
shutil.make_archive(f"/content/{sid}", "zip", "results", sid)
files.download(f"/content/{sid}.zip")